# XRF55 Bench — Ablation Ladder: TF-Mamba → WavDualMamba

Đi từ TF-Mamba (S0) đến WavDualMamba (S6), **mỗi bậc đổi đúng 1 thứ**.
Tổng các Δ giữa các bậc liên tiếp = đúng gap giữa 2 baseline.

Thứ tự thêm khối: **CNN → BiMamba → AttnStatPool** (front-to-back theo kiến trúc:
embedding → sequence → pooling). `kwargs` ở Cell 3 là **tích luỹ** — bậc sau gồm
mọi flag của bậc trước.

| Bậc | Thay đổi (so với bậc trước) | Flag cộng thêm | Trạng thái |
|---|---|---|---|
| **S0** | TF-Mamba baseline (haar {HL,LH}) | — | chạy ở đây (re-run đồng bộ code; headline cũ 89.16/86.39) |
| **S1** | + CNN front-end WavDualMamba (kernel per-subband (3,7)/(7,3) + dilated TFBlocks) | `use_cnn=True` | chạy ở đây |
| **S1.1** | (nhánh isolate) + CHỈ gated BiMamba — không CNN, không AttnStat | `mamba='bi'` (trên S0) | đo riêng BiMamba |
| **S1.2** | (nhánh isolate) + CHỈ AttnStatPool — không CNN, không BiMamba | `pool='attnstat'` (trên S0) | đo riêng pooling |
| **S2** | + gated BiMamba×2 d_state=32 (thay uni×3 d_state=16) | `+ mamba='bi'` | chạy ở đây |
| **S3** | + AttnStatPool (GAP → attentive mean+std) | `+ pool='attnstat'` | chạy ở đây |
| **S4** | dọn chi tiết (bỏ PE/proj_s3, fusion convex zero-init, head WDM) ≡ **WavDualMamba trên haar** | `wavdualmamba_haar` | chạy ở đây |
| **S5** | haar → db4 (kiến trúc y nguyên) | `wavdualmamba` + `subbands=('HL','LH')` | chạy ở đây |
| **S6** | + branch LL = WavDualMamba đầy đủ | `wavdualmamba` | chạy ở đây (re-run đồng bộ code; headline cũ 91.87/89.38) |

Δ đọc theo bậc: **ΔS1** = CNN (trên baseline trần) · **ΔS2** = BiMamba (khi đã có
CNN) · **ΔS3** = AttnStatPool (khi đã có CNN+BiMamba). S3 = đủ cả 3 khối.

**Chuẩn mọi run:** protocol 02* (AdamW lr=5e-4, wd=1e-3, betas=(0.9,0.95),
warmup 10ep + cosine→1e-6, 80 epochs, bs=32, grad_clip=1.0), 5 seeds
{0,4,8,17,42}, eval `last_model.pt` — y hệt các run benchmark headline.

**Cách dùng:** chọn `RUNGS` + `DATA_MODES` ở Cell 3 (mặc định = đủ 7 bậc × cả
`proc`+`raw`; cắt bớt cho lần thử nhanh), rồi Run All. Không sửa kwargs tay —
`LADDER` là nguồn sự thật duy nhất.

**Lưu ý:**
- Push code mới nhất lên GitHub trước khi chạy (Cell 2 pull repo).
- S0 & S6 re-run tại đây để 7 bậc cùng một ảnh code (artifact không pin commit);
  Cell 7 tự ưu tiên số chạy mới, chỉ fallback về headline cũ nếu bậc nào chưa chạy.
- ΔS6 kèm +~50% params (thêm nguyên branch LL) — ghi chú khi trình bày.
- ΔS2 và ΔS4 là delta GỘP (S2: hướng+depth+d_state; S4: bỏ PE/proj_s3 + fusion
  zero-init + head WDM) — block-level, không atomic. Ghi chú khi quy kết.
- S1–S3 dùng kernel WavDualMamba thật ((3,7)/(7,3)) — `subband_kernels=True` mặc
  định; nhờ vậy ΔS4 chỉ còn: bỏ PE/proj_s3, fusion zero-init, head WDM.
- Data Kaggle hiện build trước fix tráo nhãn cH/cV: cả PreprocTFMambaDataset
  (S0–S3) lẫn adapter S4 tự nhận diện qua `stats.meta.tfmamba_subband_naming`,
  đưa nội dung HL về stream_T (→ kernel (3,7)).

## Attached dataset

| Dataset name | Mount path |
|---|---|
| `xrf55-amp-dataset` | `/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/` |

In [ ]:
# Cell 1 — Install deps: mamba-ssm (models) + PyWavelets (on-the-fly Haar LL for S4.1/S4.2)
!pip install -q ninja packaging wheel
!pip install -q triton
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation
!pip install -q PyWavelets          # pywt — needed at runtime by dataset.py (Haar LL adapter)
print('Install done')

In [ ]:
# Cell 2 — Clone / update latest code from GitHub
import sys, subprocess
from pathlib import Path

CODE_PATH = Path('/kaggle/working/xrf55_benchmark')
if not CODE_PATH.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/imhoangt/xrf55_benchmark.git',
         str(CODE_PATH)],
        check=True
    )
else:
    subprocess.run(['git', 'pull'], cwd=str(CODE_PATH), check=True)

sys.path.insert(0, str(CODE_PATH))
print(f'Code path  : {CODE_PATH}')

from xrf55_bench.trainer import run
print('Import OK  : xrf55_bench.trainer.run')

In [ ]:
# Cell 3 — Ladder configuration: pick RUNGS + DATA_MODES, never edit kwargs per-run
from pathlib import Path

SEEDS      = [0, 4, 8, 17, 42]               # benchmark standard: full 5 seeds
RUNGS      = ['S0', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6']  # full ladder; e.g. ['S4.1','S4.2'] for the pooling study
DATA_MODES = ['proc', 'raw']                 # both; set ['proc'] or ['raw'] to run just one

# Cross-commit resume (Save Version mode): attach a PREVIOUS version's output via
# "Add Data", then list its outputs/ dir here. Cell 4 copies finished runs in
# (weights/zip skipped) so the resume guard skips them and Cell 7-9 see the full
# ladder. Leave empty [] for a single-session / first commit.
EXTRA_ROOTS = []   # e.g. [Path('/kaggle/input/<prev-version-output>/outputs')]

DATA_DIRS = {
    'raw':  Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/raw'),
    'proc': Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/processed'),
}

# ── The ladder — single source of truth. All 7 rungs runnable; S0/S6 re-run
# here for code parity with S1-S5 (run artifacts don't pin a git commit).
# Order: CNN -> BiMamba -> AttnStatPool. kwargs CUMULATIVE (each rung adds one
# flag on top of the previous). S3 = all three blocks.
# S4.1/S4.2 = side study: Haar 3-branch (S4 + LL branch), AttnStatPool vs GAP.
LADDER = {
    'S0': dict(model='tfmamba',
               kwargs={},
               note='TF-Mamba baseline, haar {HL,LH} - ladder anchor'),
    'S1': dict(model='tfmamba',
               kwargs={'use_cnn': True},
               note='S0 + WavDualMamba CNN front-end (SubbandStem + dilated TFBlocks)'),
    'S1.1': dict(model='tfmamba',
                 kwargs={'mamba': 'bi'},
                 note='S0 + ONLY gated BiMamba (no CNN, no AttnStat) - isolates BiMamba'),
    'S1.2': dict(model='tfmamba',
                 kwargs={'pool': 'attnstat'},
                 note='S0 + ONLY AttnStatPool (no CNN, no BiMamba) - isolates pooling'),
    'S1.2b': dict(model='tfmamba',
                  kwargs={'pool': 'attnstat', 'use_proj_s3': False},
                  note='S0 + AttnStatPool WITHOUT proj_s3+tanh - fair AttnStatPool test (S1.2b)'),
    'S2': dict(model='tfmamba',
               kwargs={'use_cnn': True, 'mamba': 'bi'},
               note='S1 + gated BiMamba x2 d_state=32 (replaces uni-Mamba x3 d_state=16)'),
    'S2.npe': dict(model='tfmamba',
                   kwargs={'use_cnn': True, 'mamba': 'bi', 'use_pos_emb': False},
                   note='S2 (CNN+BiMamba) NHUNG bo PE (use_pos_emb=False) - co lap tac dong cua PE'),
    'S3': dict(model='tfmamba',
               kwargs={'use_cnn': True, 'mamba': 'bi', 'pool': 'attnstat'},
               note='S2 + AttnStatPool (GAP -> attentive mean+std); = all three blocks'),
    'S4': dict(model='wavdualmamba_haar',
               kwargs={},
               note='full WavDualMamba arch on Haar {HL,LH} (tfmamba arrays, packed adapter)'),
    'S4.nogn': dict(model='wavdualmamba_haar',
                    kwargs={'stem_norm': False},
                    note='S4 bo GroupNorm cua SubbandStem (stem = Conv->SiLU; TFBlock pre-norm GN van con)'),
    'S4.nogn_gate': dict(model='wavdualmamba_haar',
                         kwargs={'stem_norm': False, 'fusion': 'gate'},
                         note='S4.nogn + fusion gate (per-channel) thay convex scalar - co lap gate tren nen bo stem-GN'),
    'S4.3': dict(model='wavdualmamba_haar',
             kwargs={'pool': 'gap'},
             note='Haar {HL,LH} + GAP - pairs with S4, isolates AttnStatPool at 2-branch level'),
    'S4.a': dict(model='wavdualmamba_haar',
                 kwargs={'use_post_fusion_proj': True},
                 note='S4 (AttnStat) + Linear(64->64) sau fusion, truoc pooling (proj_s3 KHONG tanh)'),
    'S4.b': dict(model='wavdualmamba_haar',
                 kwargs={'use_post_fusion_proj': True, 'pool': 'gap'},
                 note='S4.a nhung dung GAP (= S4.3 + Linear(64->64) sau fusion)'),
    'S4.c': dict(model='wavdualmamba_haar',
                 kwargs={'use_post_fusion_proj': True, 'post_fusion_proj_tanh': True, 'pool': 'gap'},
                 note='S4 backbone + FULL TF-Mamba head (Linear+tanh+GAP) = tai dung proj_s3+tanh tren WavDualMamba'),
    'S4.pe': dict(model='wavdualmamba_haar',
                  kwargs={'use_pos_emb': True},
                  note='S4 + sinusoidal PE (use_pos_emb=True) - kiem tra PE co giup tren WavDualMamba khong'),
    'S4.1': dict(model='wavdualmamba_haar3',
                 kwargs={'pool': 'attnstat'},
                 note='Haar 3-branch (S4 + LL) + AttnStatPool (WDM default)'),
    'S4.2': dict(model='wavdualmamba_haar3',
                 kwargs={'pool': 'gap'},
                 note='Haar 3-branch (S4 + LL) + GAP (no AttnStatPool)'),
    'S5': dict(model='wavdualmamba',
               kwargs={'subbands': ('HL', 'LH')},
               note='same arch, db4 {HL,LH} - wavelet swap'),
    'S6': dict(model='wavdualmamba',
               kwargs={},
               note='S5 + LL branch = full WavDualMamba (db4 {LL,HL,LH}) = headline'),
}

for r in RUNGS:
    assert r in LADDER, f'Unknown rung {r!r} - choose from {list(LADDER)}'
    print(f"{r}: model={LADDER[r]['model']:<18} kwargs={LADDER[r]['kwargs']}")
    print(f"    {LADDER[r]['note']}")
n_runs = len(RUNGS) * len(DATA_MODES) * len(SEEDS)
print(f'Seeds      : {SEEDS}  ({len(SEEDS)} seeds)')
print(f'Data modes : {DATA_MODES}')
print(f'Total      : {len(RUNGS)} rung x {len(DATA_MODES)} data x {len(SEEDS)} seed = {n_runs} trainings')
print(f'Resume from: {EXTRA_ROOTS or "(none)"}')
for dm in DATA_MODES:
    p = DATA_DIRS[dm] / 'stats.json'
    print(f"  [{'OK' if p.exists() else 'MISSING'}] {p}")

In [ ]:
# Cell 4 — Run the selected rungs (protocol 02*, identical to all benchmark runs)
import json
import shutil
import traceback
from xrf55_bench.config import TrainCfg_for_protocol

PROTOCOL    = '02'
OUTPUT_ROOT = Path('/kaggle/working/outputs')
FORCE_RERUN = False        # True = retrain even if a (rung, dm) already finished
RUN_DIRS    = []
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Cross-commit resume: import finished runs from a previous version's output
# (EXTRA_ROOTS, set in Cell 3). Copy WITHOUT weights/zip so the resume guard
# below skips them and Cell 7-9 aggregate the full ladder, without re-training.
for src_root in EXTRA_ROOTS:
    src_root = Path(src_root)
    if not src_root.exists():
        print(f'[resume] EXTRA_ROOT missing, skipped: {src_root}')
        continue
    for d in sorted(src_root.glob('abl_*_p02_*')):
        dst = OUTPUT_ROOT / d.name
        if not dst.exists():
            shutil.copytree(d, dst, ignore=shutil.ignore_patterns('*.pt', '*.zip'))
            print(f'[resume] imported {d.name}')

def _is_complete(out_dir, seeds):
    """Done only if metrics.json lists ALL requested seeds (a timeout-truncated
    run writes metrics.json with fewer seeds -> rerun)."""
    mp = out_dir / 'metrics.json'
    if not mp.exists():
        return False
    try:
        with open(mp) as f:
            done = set(map(int, json.load(f).get('per_seed', {})))
    except Exception:
        return False
    return set(seeds).issubset(done)

for rung in RUNGS:
    spec = LADDER[rung]
    for dm in DATA_MODES:
        out = OUTPUT_ROOT / f'abl_{rung.lower()}_p{PROTOCOL}_{dm}'
        if not FORCE_RERUN and _is_complete(out, SEEDS):
            print(f'[skip] {rung} [{dm}] already complete ({len(SEEDS)} seeds) -> {out.name}')
            RUN_DIRS.append(out)
            continue
        cfg = TrainCfg_for_protocol(
            PROTOCOL, seeds=tuple(SEEDS),
            num_epochs=80, betas=(0.9, 0.95), grad_clip=1.0,
            lr=5e-4, floor_lr=1e-6,
        )  # = protocol 02* — same as every benchmark headline run
        print('\n' + '#' * 70)
        print(f'##  {rung} [{dm}]  ->  {out}')
        print(f'##  {spec["note"]}')
        print('#' * 70)
        try:
            run(model_name=spec['model'], bench_dir=DATA_DIRS[dm], output_dir=out,
                train_cfg=cfg, num_workers=4, model_kwargs=spec['kwargs'])
            RUN_DIRS.append(out)
        except Exception as e:
            # One failing run (e.g. OOM) must not abort the whole batch/commit.
            print(f'[FAIL] {rung} [{dm}] -> {type(e).__name__}: {e}')
            traceback.print_exc()

print('\nDone:', [d.name for d in RUN_DIRS])

In [ ]:
# Cell 5 — Per-run results (reads abl_* from disk; survives kernel restart / timeout)
import json
from pathlib import Path

RESULT_ROOTS = [Path('/kaggle/working/outputs'),
                Path('../../outputs/kaggle_plots')]   # local layout

# Rediscover completed runs from disk (independent of Cell 4's in-memory RUN_DIRS).
RUN_DIRS = []
for root in RESULT_ROOTS:
    if root.exists():
        RUN_DIRS = sorted({p.parent for p in root.glob('abl_*_p02_*/metrics.json')})
        if RUN_DIRS:
            break

if not RUN_DIRS:
    print('No completed abl_* runs found on disk yet — run Cell 4 first.')
for out in RUN_DIRS:
    p = out / 'metrics.json'
    if not p.exists():
        print(f'[MISSING] {p}')
        continue
    with open(p) as f:
        m = json.load(f)
    s = m['summary']
    n_seeds = len(m.get('per_seed', {}))
    print(f"{out.name:<24} acc={s['test_accuracy_mean']*100:6.2f}% ± {s['test_accuracy_std']*100:4.2f}%   "
          f"f1={s['test_f1_macro_mean']*100:6.2f}% ± {s['test_f1_macro_std']*100:4.2f}%   "
          f"params={s['params_M']}M   lat={s['latency_mean_ms']}ms   ({n_seeds} seeds)")

In [ ]:
# Cell 6 — (optional) per-run inspection: plots inline + per-run zips (WITH weights)
# Main deliverable lives in Cell 7 (table/CSV) + Cell 8 (ladder plot) + Cell 9
# (consolidated zip, no weights). Both toggles default OFF to keep /kaggle/working
# light and avoid 42 inline images on a full sweep.
import shutil
from pathlib import Path
from IPython.display import Image, display, FileLink

SHOW_PLOTS    = False   # True: display each run's plots inline (heavy with many runs)
COPY_RUN_ZIPS = False   # True: also copy per-run zips (WITH weights) to /kaggle/working

# Rediscover runs from disk so this cell works after a kernel restart too.
try:
    RUN_DIRS
except NameError:
    RUN_DIRS = []
if not RUN_DIRS:
    for root in [Path('/kaggle/working/outputs'), Path('../../outputs/kaggle_plots')]:
        if root.exists():
            RUN_DIRS = sorted({p.parent for p in root.glob('abl_*_p02_*/metrics.json')})
            if RUN_DIRS:
                break

if not (SHOW_PLOTS or COPY_RUN_ZIPS):
    print('Cell 6 idle (SHOW_PLOTS / COPY_RUN_ZIPS = False).')
    print('Ladder plot -> Cell 8;  one-file download -> Cell 9 (ladder_results.zip).')

for out in RUN_DIRS:
    if SHOW_PLOTS:
        print(f'--- {out.name} ---')
        for fname in ['training_curve.png', 'confusion_matrix.png', 'seed_comparison.png']:
            p = out / 'plots' / fname
            if p.exists():
                display(Image(str(p)))
    if COPY_RUN_ZIPS:
        for src in sorted(out.glob('*.zip')):
            dst = Path('/kaggle/working') / src.name
            if src.resolve() != dst.resolve():
                shutil.copy2(src, dst)
            print(f'{src.name}  ({dst.stat().st_size / 1e6:.1f} MB)')
            display(FileLink(dst.name))

In [ ]:
# Cell 7 — Aggregate all rungs -> table + ladder_summary.csv/json (the deliverable)
# Prefers freshly-run abl_* metrics.json; BASELINES are a FALLBACK for rungs not re-run.
import csv
import json
from pathlib import Path

RESULT_ROOTS = [Path('/kaggle/working/outputs'),
                Path('../../outputs/kaggle_plots')]   # local layout
SUMMARY_ROOT = next((r for r in RESULT_ROOTS if r.exists()), RESULT_ROOTS[0])
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

# Fallback headline numbers (5 seeds, protocol 02*) — used only if a rung has
# no fresh run on disk. Re-running S0/S6 overrides these automatically.
BASELINES = {
    ('S0', 'proc'): dict(acc=0.891616, acc_std=0.006674, f1=0.890964, f1_std=0.006885, params=0.219),
    ('S0', 'raw'):  dict(acc=0.863940, acc_std=0.004787, f1=0.863887, f1_std=0.005055, params=0.219),
    ('S6', 'proc'): dict(acc=0.918687, acc_std=0.001720, f1=0.917708, f1_std=0.001792, params=0.588),
    ('S6', 'raw'):  dict(acc=0.893838, acc_std=0.002378, f1=0.892546, f1_std=0.002310, params=0.588),
}
RUNG_ORDER = ['S0', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6']
RUNG_DESC  = {
    'S0': 'TF-Mamba baseline (haar HL+LH)',
    'S1': '+ CNN front-end',
    'S2': '+ gated BiMamba',
    'S3': '+ AttnStatPool',
    'S4': '= WavDualMamba arch (haar)',
    'S5': 'haar -> db4',
    'S6': '+ LL = WavDualMamba (headline)',
}

def _load(p):
    with open(p) as f:
        s = json.load(f)['summary']
    return dict(acc=s['test_accuracy_mean'], acc_std=s['test_accuracy_std'],
                f1=s['test_f1_macro_mean'], f1_std=s['test_f1_macro_std'],
                params=s['params_M'])

def find_metrics(rung, dm):
    # Prefer a fresh run on disk; fall back to headline BASELINES if not re-run.
    for root in RESULT_ROOTS:
        if not root.exists():
            continue
        hits = (sorted(root.glob(f'abl_{rung.lower()}_p02_{dm}*/metrics.json'))
                + sorted(root.glob(f'abl_{rung.lower()}_p02_{dm}*/results_summary/metrics.json')))
        if hits:
            return _load(hits[-1])   # newest if several (timestamped names sort)
    return BASELINES.get((rung, dm))

# Build aggregate rows (reused by the plot + bundle cells) and print the table.
# Numeric fields rounded so the CSV/JSON deliverable is clean (no float noise).
LADDER_ROWS = []
for dm in ['proc', 'raw']:
    print(f'\n======== Ladder [{dm}] ========')
    print(f"{'Rung':<5} {'Change':<34} {'Accuracy':>16} {'dAcc':>7} {'F1 macro':>16} {'Params':>8}")
    prev = None
    for rung in RUNG_ORDER:
        m = find_metrics(rung, dm)
        if m is None:
            print(f'{rung:<5} {RUNG_DESC[rung]:<34} {"- not run -":>16}')
            prev = None   # break the delta chain at a gap
            continue
        dacc = (m['acc'] - prev) if prev is not None else None
        LADDER_ROWS.append(dict(rung=rung, data_mode=dm, change=RUNG_DESC[rung],
                                acc=round(m['acc'], 6), acc_std=round(m['acc_std'], 6),
                                f1=round(m['f1'], 6), f1_std=round(m['f1_std'], 6),
                                params_M=round(m['params'], 3),
                                dAcc=(round(dacc, 6) if dacc is not None else None)))
        d   = f'{dacc * 100:+.2f}' if dacc is not None else ''
        acc = f"{m['acc']*100:5.2f}% +- {m['acc_std']*100:4.2f}%"
        f1  = f"{m['f1']*100:5.2f}% +- {m['f1_std']*100:4.2f}%"
        print(f'{rung:<5} {RUNG_DESC[rung]:<34} {acc:>16} {d:>7} {f1:>16} {m["params"]:>7.3f}M')
        prev = m['acc']
print('Note: dS6 includes +~50% params (extra LL branch backbone).')

# Persist machine-readable aggregate — the paper deliverable (single source).
csv_path = SUMMARY_ROOT / 'ladder_summary.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['rung', 'data_mode', 'change', 'acc', 'acc_std',
                                      'f1', 'f1_std', 'params_M', 'dAcc'])
    w.writeheader()
    w.writerows(LADDER_ROWS)
with open(SUMMARY_ROOT / 'ladder_summary.json', 'w') as f:
    json.dump(LADDER_ROWS, f, indent=2)
print(f'\nSaved: {csv_path}  ({len(LADDER_ROWS)} rows)')

In [ ]:
# Cell 8 — Ladder visualization: accuracy +- std across S0->S6 (proc & raw), dAcc per step
import matplotlib.pyplot as plt

_idx  = {r: i for i, r in enumerate(RUNG_ORDER)}
STYLE = {'proc': dict(color='#2a6fdb', marker='o'),
         'raw':  dict(color='#e8761b', marker='s')}

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_axisbelow(True)
ax.grid(axis='y', alpha=0.30, linestyle='--')
ax.grid(axis='x', alpha=0.12)

allv = []
for dm in ['proc', 'raw']:
    sub = [r for r in LADDER_ROWS if r['data_mode'] == dm]
    if not sub:
        continue
    c  = STYLE[dm]['color']
    xs = [_idx[r['rung']] for r in sub]
    ys = [r['acc'] * 100 for r in sub]
    es = [r['acc_std'] * 100 for r in sub]
    lo = [y - e for y, e in zip(ys, es)]
    hi = [y + e for y, e in zip(ys, es)]
    allv += lo + hi
    ax.fill_between(xs, lo, hi, color=c, alpha=0.15, linewidth=0)
    ax.plot(xs, ys, color=c, marker=STYLE[dm]['marker'], markersize=7,
            linewidth=2.4, zorder=3,
            label=f"{dm}   (S0→S6: {ys[-1] - ys[0]:+.2f} pts)")
    # dAcc at each segment midpoint — the ladder's key message, boxed to stay readable
    for i in range(1, len(sub)):
        if sub[i]['dAcc'] is None:
            continue
        xm, ym = (xs[i-1] + xs[i]) / 2, (ys[i-1] + ys[i]) / 2
        ax.annotate(f"+{sub[i]['dAcc']*100:.2f}", (xm, ym),
                    ha='center', va='center', fontsize=8, color=c, zorder=4,
                    bbox=dict(boxstyle='round,pad=0.22', fc='white', ec=c, lw=1, alpha=0.92))
    # absolute accuracy anchored at the two endpoints only (keeps it uncluttered)
    for i in (0, len(sub) - 1):
        ax.annotate(f"{ys[i]:.2f}%", (xs[i], ys[i]), textcoords='offset points',
                    xytext=(0, 11 if dm == 'proc' else -13), ha='center',
                    fontsize=9, fontweight='bold', color=c)

ax.set_xticks(list(_idx.values()))
ax.set_xticklabels([f"{r}\n{RUNG_DESC[r].split('(')[0].strip()}" for r in RUNG_ORDER],
                   fontsize=8.5)
if allv:
    ax.set_ylim(min(allv) - 0.6, max(allv) + 1.2)   # headroom for the upper-left legend
ax.set_xlabel('Ablation rung (cumulative changes)', fontsize=11)
ax.set_ylabel('Test accuracy (%)', fontsize=11)
ax.set_title('TF-Mamba → WavDualMamba ablation ladder', fontsize=13, fontweight='bold')
ax.legend(title='data mode', fontsize=10, framealpha=0.9, loc='upper left')
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
png_path = SUMMARY_ROOT / 'ladder_acc.png'
fig.savefig(png_path, dpi=160, bbox_inches='tight')
print('Saved:', png_path)
plt.show()

In [ ]:
# Cell 9 — One consolidated download: ladder_results.zip (summary + per-run metrics/plots, NO weights)
# Per-run zips (with weights) still exist under each abl_*/ for full reproducibility.
import shutil
import zipfile
from datetime import datetime
from IPython.display import FileLink, display

ts     = datetime.now().strftime('%m%d_%H%M%S')
bundle = SUMMARY_ROOT / f'ladder_results_{ts}.zip'
n_runs = 0
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['ladder_summary.csv', 'ladder_summary.json', 'ladder_acc.png']:
        p = SUMMARY_ROOT / fname
        if p.exists():
            zf.write(p, fname)
    for root in RESULT_ROOTS:
        if not root.exists():
            continue
        for d in sorted(root.glob('abl_*_p02_*')):
            if not d.is_dir():
                continue
            for rel in ['metrics.json', 'run_config.json']:
                p = d / rel
                if p.exists():
                    zf.write(p, f'{d.name}/{rel}')
            pdir = d / 'plots'
            if pdir.exists():
                for png in sorted(pdir.glob('*.png')):
                    zf.write(png, f'{d.name}/plots/{png.name}')
            n_runs += 1
        break   # only the first existing root

print(f'Bundle: {bundle}  ({bundle.stat().st_size/1e6:.1f} MB, {n_runs} runs, no weights)')
dst = Path('/kaggle/working') / bundle.name
if bundle.resolve() != dst.resolve():
    shutil.copy2(bundle, dst)
display(FileLink(dst.name))

In [ ]:
# Cell 10 — S4-family comparison: Haar 2/3-branch x pooling, + db4 (S5/S6) if present.
# Reads abl_* from disk (restart-safe). Answers: LL-on-Haar, AttnStatPool vs GAP, Haar vs db4.
import json
from pathlib import Path

RESULT_ROOTS = [Path('/kaggle/working/outputs'), Path('../../outputs/kaggle_plots')]

def _summ(stem, dm):
    for root in RESULT_ROOTS:
        if not root.exists():
            continue
        hits = (sorted(root.glob(f'{stem}_p02_{dm}*/metrics.json'))
                + sorted(root.glob(f'{stem}_p02_{dm}*/results_summary/metrics.json')))
        if hits:
            return json.load(open(hits[-1]))['summary']
    return None

VARIANTS = [
    ('S4    Haar-2  attnstat',       'abl_s4'),
    ('S4.nogn Haar-2 attnstat no-stemGN','abl_s4.nogn'),
    ('S4.nogn_gate Haar-2 no-stemGN+gate','abl_s4.nogn_gate'),
    ('S4.3  Haar-2  gap',           'abl_s4.3'),
    ('S4.a  Haar-2  attnstat+proj', 'abl_s4.a'),
    ('S4.b  Haar-2  gap+proj',      'abl_s4.b'),
    ('S4.c  Haar-2  gap+proj+tanh', 'abl_s4.c'),
    ('S4.pe Haar-2  attnstat+PE',   'abl_s4.pe'),
    ('S4.1  Haar-3  attnstat',      'abl_s4.1'),
    ('S4.2  Haar-3  gap',           'abl_s4.2'),
    ('S5    db4-2   attnstat',      'abl_s5'),
    ('S6    db4-3   attnstat',      'abl_s6'),
]
for dm in ['proc', 'raw']:
    print(f'\n======== S4-family [{dm}] ========')
    a = {}
    for label, stem in VARIANTS:
        s = _summ(stem, dm)
        if s is None:
            print(f'{label:<26} - not run -')
            continue
        a[stem] = s['test_accuracy_mean']
        print(f"{label:<26} {s['test_accuracy_mean']*100:6.2f}% +- {s['test_accuracy_std']*100:4.2f}%"
              f"   params={s['params_M']}M")
    def dd(x, y, desc):
        v = f"{(a[x]-a[y])*100:+.2f} pts" if x in a and y in a else "(need both runs)"
        print(f"   {desc:<32} {v}")
    print('   ---- contrasts ----')
    dd('abl_s4.1', 'abl_s4',   'LL on Haar       (S4.1 - S4)')
    dd('abl_s4.2', 'abl_s4.1', 'GAP vs AttnStat  (S4.2 - S4.1)')
    dd('abl_s5',   'abl_s4',   'db4 vs Haar @2br (S5 - S4)')
    dd('abl_s6',   'abl_s4.1', 'db4 vs Haar @3br (S6 - S4.1)')
    dd('abl_s4.a', 'abl_s4',   '+proj @AttnStat  (S4.a - S4)')
    dd('abl_s4.b', 'abl_s4.3', '+proj @GAP       (S4.b - S4.3)')
    dd('abl_s4.c', 'abl_s4.b', '+tanh @GAP       (S4.c - S4.b)')
    dd('abl_s4.nogn','abl_s4',  'bo stem-GN      (S4.nogn - S4)')
    dd('abl_s4.nogn_gate','abl_s4.nogn',  'gate vs convex  (S4.nogn_gate - S4.nogn)')
    dd('abl_s4.pe', 'abl_s4',  '+PE @AttnStat    (S4.pe - S4)')